In [1]:
import os
os.chdir("/Users/alimurad/Desktop/projects/rag-pgvector/rag_pgvector")
os.listdir()

['.DS_Store', '__init__.py', 'exps', 'src']

In [2]:
from src.utils.utils import notebook_line_magic
notebook_line_magic()

## Database Connection

In [3]:
from dotenv import load_dotenv

import pandas as pd
from src.utils.db_utils import query_db, create_embedding_table, load_embeddings

load_dotenv()

True

In [19]:
from src.data.processing import load_pdf

pages = load_pdf("src/data/cava10k.pdf")
table_pages = [p["page"] for p in pages if p["has_table"]]

In [12]:
from src.utils.utils import get_token_provider
from src.models.embeddings import generate_batch_embeddings
from src.models.chat_completion import chat_completion

token_provider = get_token_provider()

### Chunking

In [33]:
create_embedding_table(
    schema=os.getenv("PG_DBNAME"),
    table="embeddings",
    embedding_dim=786
)

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdfplumber import page


def chunk_text(text: str, chunk_size: int = 512, chunk_overlap: int = 64) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_text(text)


def load_batch_embeddings(token_provider, pages, chunk_size=512, chunk_overlap=64):
    chunks = [
        {   
            "page": page['page'],
            "chunks": chunk_text(page['text'] + "\n" + page['table_content'], chunk_size, chunk_overlap)
        } for page in pages
    ]
    for temp_chunks in chunks:
        embeddings = generate_batch_embeddings(
            token_provider=token_provider, 
            texts=temp_chunks['chunks'], 
            page=temp_chunks['page'], 
            dimensions=786
        )
        load_embeddings(
            schema=os.getenv("PG_DBNAME"),
            table="embeddings",
            df=pd.DataFrame(embeddings)
        )

In [35]:
load_batch_embeddings(
    token_provider, 
    pages, 
    chunk_size=512, 
    chunk_overlap=64
)